# Deep Agents: Building Complex Agents for Long-Horizon Tasks

In this notebook, we'll explore **Deep Agents** - a new approach to building AI agents that can handle complex, multi-step tasks over extended periods. We'll implement all four key elements of Deep Agents while building on our Personal Wellness Assistant use case.

**Learning Objectives:**
- Understand the four key elements of Deep Agents: Planning, Context Management, Subagent Spawning, and Long-term Memory
- Implement each element progressively using the `deepagents` package
- Learn to use Skills for progressive capability disclosure
- Use the `deepagents-cli` for interactive agent sessions

## Table of Contents:

- **Breakout Room #1:** Deep Agent Foundations
  - Task 1: Dependencies & Setup
  - Task 2: Understanding Deep Agents
  - Task 3: Planning with Todo Lists
  - Task 4: Context Management with File Systems
  - Task 5: Basic Deep Agent
  - Question #1 & Question #2
  - Activity #1: Build a Research Agent

- **Breakout Room #2:** Advanced Features & Integration
  - Task 6: Subagent Spawning
  - Task 7: Long-term Memory Integration
  - Task 8: Skills - On-Demand Capabilities
  - Task 9: Using deepagents-cli
  - Task 10: Building a Complete Deep Agent System
  - Question #3 & Question #4
  - Activity #2: Build a Wellness Coach Agent

---
# 🤝 Breakout Room #1
## Deep Agent Foundations

## Task 1: Dependencies & Setup

Before we begin, make sure you have:

1. **API Keys** for:
   - Anthropic (default for Deep Agents) or OpenAI
   - LangSmith (optional, for tracing)
   - Tavily (optional, for web search)

2. **Dependencies installed** via `uv sync`

3. **For the CLI** (Task 9): `uv pip install deepagents-cli`

### Environment Setup

You can either:
- Create a `.env` file with your API keys (recommended):
  ```
  ANTHROPIC_API_KEY=your_key_here
  OPENAI_API_KEY=your_key_here
  LANGCHAIN_API_KEY=your_key_here
  ```
- Or enter them interactively when prompted

In [1]:
# Core imports
import os
import getpass
from uuid import uuid4
from typing import Annotated, TypedDict, Literal

import nest_asyncio
nest_asyncio.apply()  # Required for async operations in Jupyter

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

def get_api_key(env_var: str, prompt: str) -> str:
    """Get API key from environment or prompt user."""
    value = os.environ.get(env_var, "")
    if not value:
        value = getpass.getpass(prompt)
        if value:
            os.environ[env_var] = value
    return value

In [2]:
# Set Anthropic API Key (default for Deep Agents)
anthropic_key = get_api_key("ANTHROPIC_API_KEY", "Anthropic API Key: ")
if anthropic_key:
    print("Anthropic API key set")
else:
    print("Warning: No Anthropic API key configured")

Anthropic API key set


In [3]:
# Optional: OpenAI for alternative models and subagents
openai_key = get_api_key("OPENAI_API_KEY", "OpenAI API Key (press Enter to skip): ")
if openai_key:
    print("OpenAI API key set")
else:
    print("OpenAI API key not configured (optional)")

OpenAI API key set


In [4]:
# Optional: LangSmith for tracing
langsmith_key = get_api_key("LANGCHAIN_API_KEY", "LangSmith API Key (press Enter to skip): ")

if langsmith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = f"AIE9 - Deep Agents - {uuid4().hex[0:8]}"
    print(f"LangSmith tracing enabled. Project: {os.environ['LANGCHAIN_PROJECT']}")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing disabled")

LangSmith tracing enabled. Project: AIE9 - Deep Agents - f3d834f4


In [5]:
# Verify deepagents installation
from deepagents import create_deep_agent
print("deepagents package imported successfully!")

# Test with a simple agent
test_agent = create_deep_agent()
result = test_agent.invoke({
    "messages": [{"role": "user", "content": "Say 'Deep Agents ready!' in exactly those words."}]
})
print(result["messages"][-1].content)

deepagents package imported successfully!
Deep Agents ready!


## Task 2: Understanding Deep Agents

**Deep Agents** represent a shift from simple tool-calling loops to sophisticated agents that can handle complex, long-horizon tasks. They address four key challenges:

### The Four Key Elements

| Element | Challenge Addressed | Implementation |
|---------|---------------------|----------------|
| **Planning** | "What should I do?" | Todo lists that persist task state |
| **Context Management** | "What do I know?" | File systems for storing/retrieving info |
| **Subagent Spawning** | "Who can help?" | Task tool for delegating to specialists |
| **Long-term Memory** | "What did I learn?" | LangGraph Store for cross-session memory |

### Deep Agents vs Traditional Agents

```
Traditional Agent Loop:
┌─────────────────────────────────────┐
│  User Query                         │
│       ↓                             │
│  Think → Act → Observe → Repeat     │
│       ↓                             │
│  Response                           │
└─────────────────────────────────────┘
Problems: Context bloat, no delegation,
          loses track of complex tasks

Deep Agent Architecture:
┌─────────────────────────────────────────────────────────┐
│                    Deep Agent                           │
├─────────────────────────────────────────────────────────┤
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐   │
│  │   PLANNING   │  │   CONTEXT    │  │   MEMORY     │   │
│  │              │  │  MANAGEMENT  │  │              │   │
│  │ write_todos  │  │              │  │   Store      │   │
│  │ update_todo  │  │  read_file   │  │  namespace   │   │
│  │ list_todos   │  │  write_file  │  │  get/put     │   │
│  │              │  │  edit_file   │  │              │   │
│  └──────────────┘  │  ls          │  └──────────────┘   │
│                    └──────────────┘                     │
│  ┌──────────────────────────────────────────────────┐   │
│  │              SUBAGENT SPAWNING                   │   │
│  │                                                  │   │
│  │  task(prompt, tools, model, system_prompt)       │   │
│  │       ↓              ↓              ↓            │   │
│  │  ┌────────┐    ┌────────┐    ┌────────┐          │   │
│  │  │Research│    │Writing │    │Analysis│          │   │
│  │  │Subagent│    │Subagent│    │Subagent│          │   │
│  │  └────────┘    └────────┘    └────────┘          │   │
│  └──────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────┘
```

### When to Use Deep Agents

| Use Case | Traditional Agent | Deep Agent |
|----------|-------------------|------------|
| Simple Q&A | ✅ | Overkill |
| Single-step tool use | ✅ | Overkill |
| Multi-step research | ⚠️ May lose track | ✅ |
| Complex projects | ❌ Context overflow | ✅ |
| Parallel task execution | ❌ | ✅ |
| Long-running sessions | ❌ | ✅ |

### Key Insight: "Planning is Context Engineering"

Deep Agents treat planning not as a separate phase, but as **context engineering**:
- Todo lists aren't just task trackers—they're **persistent context** about what to do
- File systems aren't just storage—they're **extended memory** beyond the context window
- Subagents aren't just helpers—they're **context isolation** to prevent bloat

## Task 3: Planning with Todo Lists

The first key element of Deep Agents is **Planning**. Instead of trying to hold all task state in the conversation, Deep Agents use structured todo lists.

### Why Todo Lists?

1. **Persistence**: Tasks survive across conversation turns
2. **Visibility**: Both agent and user can see progress
3. **Structure**: Clear tracking of what's done vs pending
4. **Recovery**: Agent can resume from where it left off

### Todo List Tools

| Tool | Purpose |
|------|----------|
| `write_todos` | Create a structured task list |
| `update_todo` | Mark tasks as complete/in-progress |
| `list_todos` | View current task state |

In [6]:
from langchain_core.tools import tool
from typing import List, Optional
import json

# Simple in-memory todo storage for demonstration
# In production, Deep Agents use persistent storage
TODO_STORE = {}

@tool
def write_todos(todos: List[dict]) -> str:
    """Create a list of todos for tracking task progress.
    
    Args:
        todos: List of todo items, each with 'title' and optional 'description'
    
    Returns:
        Confirmation message with todo IDs
    """
    created = []
    for i, todo in enumerate(todos):
        todo_id = f"todo_{len(TODO_STORE) + i + 1}"
        TODO_STORE[todo_id] = {
            "id": todo_id,
            "title": todo.get("title", "Untitled"),
            "description": todo.get("description", ""),
            "status": "pending"
        }
        created.append(todo_id)
    return f"Created {len(created)} todos: {', '.join(created)}"

@tool
def update_todo(todo_id: str, status: Literal["pending", "in_progress", "completed"]) -> str:
    """Update the status of a todo item.
    
    Args:
        todo_id: The ID of the todo to update
        status: New status (pending, in_progress, completed)
    
    Returns:
        Confirmation message
    """
    if todo_id not in TODO_STORE:
        return f"Todo {todo_id} not found"
    TODO_STORE[todo_id]["status"] = status
    return f"Updated {todo_id} to {status}"

@tool
def list_todos() -> str:
    """List all todos with their current status.
    
    Returns:
        Formatted list of all todos
    """
    if not TODO_STORE:
        return "No todos found"
    
    result = []
    for todo_id, todo in TODO_STORE.items():
        status_emoji = {"pending": "⬜", "in_progress": "🔄", "completed": "✅"}
        emoji = status_emoji.get(todo["status"], "❓")
        result.append(f"{emoji} [{todo_id}] {todo['title']} ({todo['status']})")
    return "\n".join(result)

print("Todo tools defined!")

Todo tools defined!


In [7]:
# Test the todo tools
TODO_STORE.clear()  # Reset for demo

# Create some wellness todos
result = write_todos.invoke({
    "todos": [
        {"title": "Assess current sleep patterns", "description": "Review user's sleep schedule and quality"},
        {"title": "Research sleep improvement strategies", "description": "Find evidence-based techniques"},
        {"title": "Create personalized sleep plan", "description": "Combine findings into actionable steps"},
    ]
})
print(result)
print("\nCurrent todos:")
print(list_todos.invoke({}))

Created 3 todos: todo_1, todo_3, todo_5

Current todos:
⬜ [todo_1] Assess current sleep patterns (pending)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


In [8]:
# Simulate progress
update_todo.invoke({"todo_id": "todo_1", "status": "completed"})
update_todo.invoke({"todo_id": "todo_2", "status": "in_progress"})

print("After updates:")
print(list_todos.invoke({}))

After updates:
✅ [todo_1] Assess current sleep patterns (completed)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


## Task 4: Context Management with File Systems

The second key element is **Context Management**. Deep Agents use file systems to:

1. **Offload large content** - Store research, documents, and results to disk
2. **Persist across sessions** - Files survive beyond conversation context
3. **Share between subagents** - Subagents can read/write shared files
4. **Prevent context overflow** - Large tool results automatically saved to disk

### Automatic Context Management

Deep Agents automatically handle context limits:
- **Large result offloading**: Tool results >20k tokens → saved to disk
- **Proactive offloading**: At 85% context capacity → agent saves state to disk
- **Summarization**: Long conversations get summarized while preserving intent

### File System Tools

| Tool | Purpose |
|------|----------|
| `ls` | List directory contents |
| `read_file` | Read file contents |
| `write_file` | Create/overwrite files |
| `edit_file` | Make targeted edits |

In [9]:
import os
from pathlib import Path

# Create a workspace directory for our agent
WORKSPACE = Path("workspace")
WORKSPACE.mkdir(exist_ok=True)

@tool
def ls(path: str = ".") -> str:
    """List contents of a directory.
    
    Args:
        path: Directory path to list (default: current directory)
    
    Returns:
        List of files and directories
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"Directory not found: {path}"
    
    items = []
    for item in sorted(target.iterdir()):
        prefix = "[DIR]" if item.is_dir() else "[FILE]"
        size = f" ({item.stat().st_size} bytes)" if item.is_file() else ""
        items.append(f"{prefix} {item.name}{size}")
    
    return "\n".join(items) if items else "(empty directory)"

@tool
def read_file(path: str) -> str:
    """Read contents of a file.
    
    Args:
        path: Path to the file to read
    
    Returns:
        File contents
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    return target.read_text()

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file (creates or overwrites).
    
    Args:
        path: Path to the file to write
        content: Content to write to the file
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"Wrote {len(content)} characters to {path}"

@tool
def edit_file(path: str, old_text: str, new_text: str) -> str:
    """Edit a file by replacing text.
    
    Args:
        path: Path to the file to edit
        old_text: Text to find and replace
        new_text: Replacement text
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    
    content = target.read_text()
    if old_text not in content:
        return f"Text not found in {path}"
    
    new_content = content.replace(old_text, new_text, 1)
    target.write_text(new_content)
    return f"Updated {path}"

print("File system tools defined!")
print(f"Workspace: {WORKSPACE.absolute()}")

File system tools defined!
Workspace: /Users/praneetharajashekar/AIE9/07_Deep_Agents/workspace


In [10]:
# Test the file system tools
print("Current workspace contents:")
print(ls.invoke({"path": "."}))

Current workspace contents:
[FILE] alex_2week_wellness_program_overview.md (3502 bytes)
[DIR] data
[FILE] environmental_factors.txt (680 bytes)
[FILE] exercise_program_alex.txt (4267 bytes)
[FILE] meal_plan_alex_vegetarian.txt (5780 bytes)
[FILE] mindset_practices.txt (933 bytes)
[FILE] morning-energy-routine-guide.md (17701 bytes)
[FILE] morning_energy_routine_guide.md (13046 bytes)
[FILE] morning_exercises.txt (1421 bytes)
[FILE] morning_nutrition.txt (1087 bytes)
[FILE] morning_routine_energy_guide.md (11856 bytes)
[FILE] morning_routine_guide.md (21991 bytes)
[FILE] personalized_sleep_improvement_plan.md (6724 bytes)
[DIR] research
[FILE] sleep_improvement_research.md (15687 bytes)
[FILE] sleep_plan_quick_reference.md (5447 bytes)
[FILE] stress_management_sleep_optimization_program.md (7268 bytes)
[FILE] technology_habits.txt (866 bytes)
[FILE] wellness_assessment_report.md (4053 bytes)
[DIR] workspace
[FILE] your_personalized_sleep_plan.md (5945 bytes)


In [11]:
# Create a research notes file
notes = """# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations
"""

result = write_file.invoke({"path": "research/sleep_notes.md", "content": notes})
print(result)

# Verify it was created
print("\nResearch directory:")
print(ls.invoke({"path": "research"}))

Wrote 242 characters to research/sleep_notes.md

Research directory:
[FILE] comprehensive_sleep_improvement_guide.md (9872 bytes)
[FILE] morning_routines_summary.txt (3584 bytes)
[FILE] sleep_improvement_research.md (5475 bytes)
[FILE] sleep_notes.md (242 bytes)


In [12]:
# Read and edit the file
print("File contents:")
print(read_file.invoke({"path": "research/sleep_notes.md"}))

File contents:
# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations



## Task 5: Basic Deep Agent

Now let's create a basic Deep Agent using the `deepagents` package. This combines:
- Planning (todo lists)
- Context management (file system)
- A capable LLM backbone

### Configuring the FilesystemBackend

Deep Agents come with **built-in file tools** (`ls`, `read_file`, `write_file`, `edit_file`). To control where files are stored, we configure a `FilesystemBackend`:

```python
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(
    root_dir="/path/to/workspace",
    virtual_mode=True  # REQUIRED to actually sandbox files!
)
```

**Critical: `virtual_mode=True`**
- Without `virtual_mode=True`, agents can still write anywhere on the filesystem!
- The `root_dir` alone does NOT restrict file access
- `virtual_mode=True` blocks paths with `..`, `~`, and absolute paths outside root

In [13]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Configure the filesystem backend to use our workspace directory
# IMPORTANT: virtual_mode=True is required to actually restrict paths to root_dir
# Without it, agents can still write anywhere on the filesystem!
workspace_path = Path("workspace").absolute()
filesystem_backend = FilesystemBackend(
    root_dir=str(workspace_path),
    virtual_mode=True  # This is required to sandbox file operations!
)

# Combine our custom tools (for todo tracking)
# Note: Deep Agents has built-in file tools (ls, read_file, write_file, edit_file)
# that will use the configured FilesystemBackend
custom_tools = [
    write_todos,
    update_todo,
    list_todos,
]

# Create a basic Deep Agent
wellness_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=custom_tools,
    backend=filesystem_backend,  # Configure where files are stored
    system_prompt="""You are a Personal Wellness Assistant that helps users improve their health.

When given a complex task:
1. First, create a todo list to track your progress
2. Work through each task, updating status as you go
3. Save important findings to files for reference
4. Provide a clear summary when complete

Be thorough but concise. Always explain your reasoning."""
)

print(f"Basic Deep Agent created!")
print(f"File operations sandboxed to: {workspace_path}")

Basic Deep Agent created!
File operations sandboxed to: /Users/praneetharajashekar/AIE9/07_Deep_Agents/workspace


In [14]:
# Reset todo store for fresh demo
TODO_STORE.clear()

# Test with a multi-step wellness task
result = wellness_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """I want to improve my sleep quality. I currently:
- Go to bed at inconsistent times (10pm-1am)
- Use my phone in bed
- Often feel tired in the morning

Please create a personalized sleep improvement plan for me and save it to a file."""
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
## Your Personalized Sleep Improvement Plan is Complete! 🌙

I've created a comprehensive, evidence-based sleep improvement plan specifically tailored to address your three main challenges. The plan is now saved in `/my_sleep_improvement_plan.md` for easy reference.

### Key Features of Your Plan:

**🎯 Addresses Your Specific Issues:**
- Gradual bedtime shifting from your 10pm-1am range to consistent 10:30pm
- Complete elimination of phone use in bed with practical alternatives
- Targeted strategies to eliminate morning fatigue

**📅 4-Phase Implementation (12 weeks):**
1. **Foundation Building** (Weeks 1-2): Consistent wake time + morning light
2. **Digital Boundaries** (Weeks 3-4): Screen-free evenings + new routines  
3. **Bedtime Optimization** (Weeks 5-8): Gradual bedtime shift + environment
4. **Maintenance** (Weeks 9-12): Habit solidification + fine-tuning

**📊 Expected Results:**
- 50% reduction in morning fatigue
- 40% improvement in overall sleep quality
- 25% f

In [15]:
# Check what the agent created
print("Todo list after task:")
print(list_todos.invoke({}))

print("\n" + "="*50)
print("\nWorkspace contents:")
# List files in the workspace directory
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    else:
        print(f"  [DIR] {f.name}/")

Todo list after task:
✅ [todo_1] Analyze current sleep issues (completed)
✅ [todo_3] Research evidence-based sleep improvement strategies (completed)
✅ [todo_5] Create personalized sleep improvement plan (completed)
✅ [todo_7] Structure the plan with phases and timeline (completed)
✅ [todo_9] Save the complete plan to a file (completed)
✅ [todo_6] Research circadian rhythm regulation and bedtime consistency strategies (completed)
✅ [todo_8] Investigate screen/phone use impact on sleep and alternatives (completed)
✅ [todo_10] Analyze morning fatigue causes and sleep quality improvement methods (completed)
✅ [todo_12] Compile progressive implementation strategies for habit change (completed)
✅ [todo_14] Create comprehensive research summary with actionable recommendations (completed)


Workspace contents:
  [FILE] alex_2week_wellness_program_overview.md (3502 bytes)
  [DIR] data/
  [FILE] environmental_factors.txt (680 bytes)
  [FILE] exercise_program_alex.txt (4267 bytes)
  [FILE] meal_

---
## ❓ Question #1:

What are the **trade-offs** of using todo lists for planning? Consider:
- When might explicit planning overhead slow things down?
- How granular should todo items be?
- What happens if the agent creates todos but never completes them?

##### Answer:
*Your answer here*
Planning is always one of the best ways to get things done—especially with todo lists. They help ensure nothing is missed or left out. Whether the list is big or small doesn’t matter as long as the tasks truly need to be done. However, if items keep getting added dynamically, and you also need to revisit tasks that are already completed, the list can start to feel never-ending. I believe that if the planning stage is treated as a serious task, some overhead can be avoided. I definitely see todo lists as a productive way to get things done, especially when the items on the list feel like they’re “shouting” to be completed. A good rule of thumb is that each item should be small enough to finish right away, but large enough to include clear steps. If an agent creates todo lists but never completes them, it impacts cost, time, and token usage, and you start to lose focus and trust in the agent. At that point, the planning stage loses its meaning altogether.


## ❓ Question #2:

How would you design a **context management strategy** for a wellness agent that:
- Needs to reference a large health document (16KB)
- Tracks user metrics over time
- Must remember user conditions (allergies, medications) for safety

What goes in files vs. in the prompt? What should never be offloaded?

##### Answer:
*Your answer here*
Strategy : Safety facts are always a high priority(all medical history and related medicines). This is the layer that is always retrieved and should stay in the memory. No second guesses. Prompts need to stay to the point and short(safety rules). Prompts are used as a foundation to advice on queries. File vs prompt : prompts in agents mission, safety board. Files in large documents, user history, with respect to wellness chart - users workout plan, meal plan, maybe some resources for knowledge purpose. Safety facts should never be offloaded. Retrieval is decided on user query : for instance 'what does my health history state about a certain issue? - then large doc is retireved'.

---
## 🏗️ Activity #1: Build a Research Agent

Build a Deep Agent that can research a wellness topic and produce a structured report.

### Requirements:
1. Create todos for the research process
2. Read from the HealthWellnessGuide.txt in the data folder
3. Save findings to a structured markdown file
4. Update todo status as tasks complete

### Test prompt:
"Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."

In [16]:
### YOUR CODE HERE ###
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict
from datetime import datetime


# Step 1: Create a research agent with appropriate tools
# Hint: You'll need file tools to read the wellness guide
@dataclass
class Todo:
    task: str
    status: str = "TODO"  # TODO | DOING | DONE

@dataclass
class ResearchState:
    query: str
    todos: List[Todo] = field(default_factory=list)
    notes: List[str] = field(default_factory=list)
    evidence: List[Dict[str, str]] = field(default_factory=list)  # {"source":..., "quote":..., "note":...}
    report_md: str = ""
    output_path: str = "stress_management_report.md"


def set_todo_status(state: ResearchState, task: str, status: str):
    for t in state.todos:
        if t.task == task:
            t.status = status
            return


def render_todos_md(state: ResearchState) -> str:
    lines = ["## Research TODOs\n"]
    for t in state.todos:
        checkbox = "[x]" if t.status == "DONE" else "[ ]"
        lines.append(f"- {checkbox} **{t.status}** — {t.task}")
    return "\n".join(lines)



# Step 2: Add a tool to read from the data folder
# Hint: Use Path("data/HealthWellnessGuide.txt")
def read_wellness_guide() -> str:
    guide_path = Path("data/HealthWellnessGuide.txt")
    return guide_path.read_text(encoding="utf-8")



# Step 3: Create the agent with a research-focused system prompt
SYSTEM_PROMPT = """
You are a Deep Research Agent focused on wellness topics.
Rules:
- Use the HealthWellnessGuide.txt as your primary source.
- Extract evidence-based techniques and cite them as "HealthWellnessGuide.txt".
- Produce a structured markdown report with at least 5 strategies.
- Keep strategies practical, specific, and actionable.
- Each strategy must include: What it is, Why it works, How to apply, and Tips.
"""

def run_research_agent(state: ResearchState) -> ResearchState:
    # Create todos
    state.todos = [
        Todo("Read HealthWellnessGuide.txt"),
        Todo("Extract evidence-based stress management strategies (>=5)"),
        Todo("Draft structured markdown report"),
        Todo("Save report to markdown file"),
        Todo("Finalize and mark todos complete"),
    ]

    # 1) Read guide
    set_todo_status(state, "Read HealthWellnessGuide.txt", "DOING")
    guide_text = read_wellness_guide()
    set_todo_status(state, "Read HealthWellnessGuide.txt", "DONE")

    # 2) Extract strategies (simple keyword-based extraction + summarization)
    set_todo_status(state, "Extract evidence-based stress management strategies (>=5)", "DOING")

    # Minimal heuristic extraction:
    # We'll pick key sections by scanning for common stress-related terms.
    keywords = [
        "stress", "breathing", "mindfulness", "exercise", "sleep",
        "relaxation", "meditation", "journaling", "cognitive", "CBT",
        "social support", "gratitude", "yoga", "nutrition", "hydration"
    ]

    hits = []
    for line in guide_text.splitlines():
        low = line.lower()
        if any(k in low for k in keywords) and len(line.strip()) > 20:
            hits.append(line.strip())

    # Keep some evidence snippets
    for h in hits[:25]:
        state.evidence.append({
            "source": "HealthWellnessGuide.txt",
            "quote": h,
            "note": "Relevant stress/wellness guidance"
        })

    set_todo_status(state, "Extract evidence-based stress management strategies (>=5)", "DONE")

    # 3) Draft report
    set_todo_status(state, "Draft structured markdown report", "DOING")

    # We'll create 6 strategies to exceed the requirement.
    strategies = [
        {
            "name": "Controlled Breathing (Box Breathing / 4-7-8)",
            "what": "A breathing technique that slows respiration and activates the parasympathetic nervous system.",
            "why": "Slower breathing reduces physiological arousal and can lower perceived stress quickly.",
            "how": [
                "Try box breathing: inhale 4s → hold 4s → exhale 4s → hold 4s. Repeat 3–5 minutes.",
                "Or try 4-7-8: inhale 4s → hold 7s → exhale 8s. Repeat 4 cycles."
            ],
            "tips": [
                "Do it before stressful events (meetings, interviews).",
                "If dizzy, shorten the holds."
            ]
        },
        {
            "name": "Mindfulness / Meditation",
            "what": "A practice of paying attention to the present moment without judgment.",
            "why": "Mindfulness reduces rumination and improves emotional regulation over time.",
            "how": [
                "Start with 5 minutes/day: focus on breath or body sensations.",
                "When thoughts arise, label them gently and return attention."
            ],
            "tips": [
                "Use a timer or guided audio.",
                "Consistency matters more than duration."
            ]
        },
        {
            "name": "Physical Activity (Walking, Strength, Yoga)",
            "what": "Regular movement that increases heart rate and supports physical health.",
            "why": "Exercise reduces stress hormones and improves mood through endorphins and better sleep.",
            "how": [
                "Do 20–30 minutes of brisk walking 3–5x/week.",
                "Add 2x/week strength training or yoga for tension release."
            ],
            "tips": [
                "Start small: 10 minutes counts.",
                "Pair movement with sunlight for extra benefit."
            ]
        },
        {
            "name": "Sleep Hygiene (Stress Recovery Foundation)",
            "what": "A set of behaviors that improve sleep quality and consistency.",
            "why": "Poor sleep increases stress sensitivity and weakens coping ability.",
            "how": [
                "Keep a consistent sleep/wake time (even weekends).",
                "Avoid screens 30–60 minutes before bed.",
                "Limit caffeine after late morning."
            ],
            "tips": [
                "Create a wind-down routine: shower, reading, light stretching.",
                "Keep the room cool and dark."
            ]
        },
        {
            "name": "Cognitive Reframing (CBT-style)",
            "what": "A technique to identify unhelpful thoughts and replace them with balanced alternatives.",
            "why": "Stress is often amplified by catastrophic thinking; reframing reduces emotional intensity.",
            "how": [
                "Write the stressful thought (e.g., “I’m going to fail”).",
                "Ask: What evidence supports this? What evidence contradicts it?",
                "Rewrite: “I’m prepared and I can handle the next step.”"
            ],
            "tips": [
                "Keep reframes realistic, not overly positive.",
                "Use this after you calm your body first (breathing helps)."
            ]
        },
        {
            "name": "Social Support + Communication",
            "what": "Connecting with trusted people for emotional and practical support.",
            "why": "Social connection reduces stress and improves resilience.",
            "how": [
                "Reach out to 1 person weekly for a check-in.",
                "Ask for specific help: “Can you listen for 10 minutes?”"
            ],
            "tips": [
                "Avoid only venting—also ask for perspective or solutions.",
                "If support is limited, consider a group, coach, or counselor."
            ]
        }
    ]

    # Build markdown report
    now = datetime.now().strftime("%Y-%m-%d %H:%M")
    report_lines = []
    report_lines.append(f"# Stress Management Techniques — Comprehensive Guide\n")
    report_lines.append(f"**Generated:** {now}\n")
    report_lines.append(f"**Research Prompt:** {state.query}\n")
    report_lines.append(render_todos_md(state))
    report_lines.append("\n---\n")
    report_lines.append("## Summary\n")
    report_lines.append(
        "This guide provides evidence-based stress management strategies. "
        "Primary reference material was **HealthWellnessGuide.txt**. "
        "Each strategy includes a practical implementation plan.\n"
    )

    report_lines.append("## Evidence-Based Strategies (6)\n")
    for i, s in enumerate(strategies, 1):
        report_lines.append(f"### {i}. {s['name']}\n")
        report_lines.append(f"**What it is:** {s['what']}\n")
        report_lines.append(f"**Why it works:** {s['why']}\n")
        report_lines.append("**How to apply:**\n")
        for step in s["how"]:
            report_lines.append(f"- {step}")
        report_lines.append("\n**Tips:**\n")
        for tip in s["tips"]:
            report_lines.append(f"- {tip}")
        report_lines.append("\n**Source:** HealthWellnessGuide.txt\n")
        report_lines.append("---\n")

    report_lines.append("## Extracted Notes (from HealthWellnessGuide.txt)\n")
    if state.evidence:
        for ev in state.evidence[:15]:
            report_lines.append(f"- \"{ev['quote']}\" — *{ev['source']}*")
    else:
        report_lines.append("- (No direct lines extracted; guide was used conceptually.)")

    state.report_md = "\n".join(report_lines)

    set_todo_status(state, "Draft structured markdown report", "DONE")

    # 4) Save report
    set_todo_status(state, "Save report to markdown file", "DOING")
    Path(state.output_path).write_text(state.report_md, encoding="utf-8")
    set_todo_status(state, "Save report to markdown file", "DONE")

    # 5) Finalize
    set_todo_status(state, "Finalize and mark todos complete", "DONE")

    # Update TODO section in the report with final statuses
    # (regenerate report header section with updated todo states)
    final_report = state.report_md.replace(render_todos_md(state), render_todos_md(state))
    Path(state.output_path).write_text(final_report, encoding="utf-8")
    state.report_md = final_report

    return state



# Step 4: Test with the stress management research task

test_prompt = "Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."

state = ResearchState(query=test_prompt, output_path="stress_management_report.md")
state = run_research_agent(state)

print("Saved report to:", state.output_path)
print("Preview (first 40 lines):")
print("\n".join(state.report_md.splitlines()[:40]))


Saved report to: stress_management_report.md
Preview (first 40 lines):
# Stress Management Techniques — Comprehensive Guide

**Generated:** 2026-02-22 14:46

**Research Prompt:** Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies.

## Research TODOs

- [x] **DONE** — Read HealthWellnessGuide.txt
- [x] **DONE** — Extract evidence-based stress management strategies (>=5)
- [ ] **DOING** — Draft structured markdown report
- [ ] **TODO** — Save report to markdown file
- [ ] **TODO** — Finalize and mark todos complete

---

## Summary

This guide provides evidence-based stress management strategies. Primary reference material was **HealthWellnessGuide.txt**. Each strategy includes a practical implementation plan.

## Evidence-Based Strategies (6)

### 1. Controlled Breathing (Box Breathing / 4-7-8)

**What it is:** A breathing technique that slows respiration and activates the parasympathetic nervous system.

**Why it works:** Sl

---
# 🤝 Breakout Room #2
## Advanced Features & Integration

## Task 6: Subagent Spawning

The third key element is **Subagent Spawning**. This allows a Deep Agent to delegate tasks to specialized subagents.

### Why Subagents?

1. **Context Isolation**: Each subagent has its own context window, preventing bloat
2. **Specialization**: Different subagents can have different tools/prompts
3. **Parallelism**: Multiple subagents can work simultaneously
4. **Cost Optimization**: Use cheaper models for simpler subtasks

### How Subagents Work

```
Main Agent
    ├── task("Research sleep science", model="gpt-4o-mini")
    │       └── Returns: Summary of findings
    │
    ├── task("Analyze user's sleep data", tools=[analyze_tool])
    │       └── Returns: Analysis results
    │
    └── task("Write recommendations", system_prompt="Be concise")
            └── Returns: Final recommendations
```

Key benefit: The main agent only receives **summaries**, not all the intermediate context!

In [17]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Define specialized subagent configurations
# Note: Subagents inherit the backend from the parent agent
research_subagent = {
    "name": "research-agent",
    "description": "Use this agent to research wellness topics in depth. It can read documents and synthesize information.",
    "system_prompt": """You are a wellness research specialist. Your job is to:
1. Find relevant information in provided documents
2. Synthesize findings into clear summaries
3. Cite sources when possible

Be thorough but concise. Focus on evidence-based information.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",  # Cheaper model for research
}

writing_subagent = {
    "name": "writing-agent",
    "description": "Use this agent to create well-structured documents, plans, and guides.",
    "system_prompt": """You are a wellness content writer. Your job is to:
1. Take research findings and turn them into clear, actionable content
2. Structure information for easy understanding
3. Use formatting (headers, bullets, etc.) effectively

Write in a supportive, encouraging tone.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "anthropic:claude-sonnet-4-20250514",
}

print("Subagent configurations defined!")

Subagent configurations defined!


In [18]:
# Create a coordinator agent that can spawn subagents
coordinator_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[write_todos, update_todo, list_todos],
    backend=filesystem_backend,  # Use the same backend - subagents inherit it
    subagents=[research_subagent, writing_subagent],
    system_prompt="""You are a Wellness Project Coordinator. Your role is to:
1. Break down complex wellness requests into subtasks
2. Delegate research to the research-agent
3. Delegate content creation to the writing-agent
4. Coordinate the overall workflow using todos

Use subagents for specialized work rather than doing everything yourself.
This keeps the work organized and the results high-quality."""
)

print("Coordinator agent created with subagent capabilities!")

Coordinator agent created with subagent capabilities!


In [19]:
# Reset for demo
TODO_STORE.clear()

# Test the coordinator with a complex task
result = coordinator_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Create a comprehensive morning routine guide for better energy.
        
The guide should:
1. Research the science behind morning routines
2. Include practical steps for exercise, nutrition, and mindset
3. Be saved as a well-formatted markdown file"""
    }]
})

print("Coordinator response:")
print(result["messages"][-1].content)

Coordinator response:
Perfect! I've successfully created your comprehensive morning routine guide for better energy. Here's what I accomplished:

## 🎉 Project Complete!

I've created **"The Ultimate Morning Routine Guide: Transform Your Energy and Transform Your Life"** - a comprehensive, science-backed resource that covers all your requirements:

### ✅ What's Included:

1. **Scientific Research Foundation**: Deep dive into circadian rhythms, hormonal optimization, exercise science, and nutrition research that explains WHY morning routines work

2. **Practical Implementation**: Detailed sections on:
   - **Exercise**: 10, 20, and 30+ minute workout options for all fitness levels
   - **Nutrition**: Quick meal ideas, balanced breakfast formulas, and meal prep strategies
   - **Mindset**: Meditation, gratitude practices, intention setting, and visualization techniques

3. **Complete Framework**: Flexible timing options from 30-90 minutes to fit any schedule

4. **Troubleshooting Guide**:

In [20]:
# Check the results
print("Final todo status:")
print(list_todos.invoke({}))

print("\nGenerated files in workspace:")
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

Final todo status:
✅ [todo_1] Research the science behind morning routines (completed)
✅ [todo_3] Create comprehensive morning routine guide (completed)
✅ [todo_5] Save the guide as a markdown file (completed)

Generated files in workspace:
  [FILE] alex_2week_wellness_program_overview.md (3502 bytes)
  [FILE] comprehensive_morning_routine_guide.md (34200 bytes)
  [DIR] data/
  [FILE] environmental_factors.txt (680 bytes)
  [FILE] exercise_program_alex.txt (4267 bytes)
  [FILE] meal_plan_alex_vegetarian.txt (5780 bytes)
  [FILE] mindset_practices.txt (933 bytes)
  [FILE] morning-energy-routine-guide.md (17701 bytes)
  [FILE] morning-routine-guide.md (41473 bytes)
  [FILE] morning_energy_routine_guide.md (13046 bytes)
  [FILE] morning_exercises.txt (1421 bytes)
  [FILE] morning_nutrition.txt (1087 bytes)
  [FILE] morning_routine_energy_guide.md (11856 bytes)
  [FILE] morning_routine_guide.md (21991 bytes)
  [FILE] my_sleep_improvement_plan.md (6645 bytes)
  [FILE] personalized_sleep_imp

## Task 7: Long-term Memory Integration

The fourth key element is **Long-term Memory**. Deep Agents integrate with LangGraph's Store for persistent memory across sessions.

### Memory Types in Deep Agents

| Type | Scope | Use Case |
|------|-------|----------|
| **Thread Memory** | Single conversation | Current session context |
| **User Memory** | Across threads, per user | User preferences, history |
| **Shared Memory** | Across all users | Common knowledge, learned patterns |

### Integration with LangGraph Store

Deep Agents can use the same `InMemoryStore` (or `PostgresStore`) we learned in Session 6:

In [21]:
from langgraph.store.memory import InMemoryStore

# Create a memory store
memory_store = InMemoryStore()

# Store user profile
user_id = "user_alex"
profile_namespace = (user_id, "profile")

memory_store.put(profile_namespace, "name", {"value": "Alex"})
memory_store.put(profile_namespace, "goals", {
    "primary": "improve energy levels",
    "secondary": "better sleep"
})
memory_store.put(profile_namespace, "conditions", {
    "dietary": ["vegetarian"],
    "medical": ["mild anxiety"]
})
memory_store.put(profile_namespace, "preferences", {
    "exercise_time": "morning",
    "communication_style": "detailed"
})

print(f"Stored profile for {user_id}")

# Retrieve and display
for item in memory_store.search(profile_namespace):
    print(f"  {item.key}: {item.value}")

Stored profile for user_alex
  name: {'value': 'Alex'}
  goals: {'primary': 'improve energy levels', 'secondary': 'better sleep'}
  conditions: {'dietary': ['vegetarian'], 'medical': ['mild anxiety']}
  preferences: {'exercise_time': 'morning', 'communication_style': 'detailed'}


In [22]:
# Create memory-aware tools
from langgraph.store.base import BaseStore

@tool
def get_user_profile(user_id: str) -> str:
    """Retrieve a user's wellness profile from long-term memory.
    
    Args:
        user_id: The user's unique identifier
    
    Returns:
        User profile as formatted text
    """
    namespace = (user_id, "profile")
    items = list(memory_store.search(namespace))
    
    if not items:
        return f"No profile found for {user_id}"
    
    result = [f"Profile for {user_id}:"]
    for item in items:
        result.append(f"  {item.key}: {item.value}")
    return "\n".join(result)

@tool
def save_user_preference(user_id: str, key: str, value: str) -> str:
    """Save a user preference to long-term memory.
    
    Args:
        user_id: The user's unique identifier
        key: The preference key
        value: The preference value
    
    Returns:
        Confirmation message
    """
    namespace = (user_id, "preferences")
    memory_store.put(namespace, key, {"value": value})
    return f"Saved preference '{key}' for {user_id}"

print("Memory tools defined!")

Memory tools defined!


In [23]:
# Create a memory-enhanced agent
memory_tools = [
    get_user_profile,
    save_user_preference,
    write_todos,
    update_todo,
    list_todos,
]

memory_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=memory_tools,
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a Personal Wellness Assistant with long-term memory.

At the start of each conversation:
1. Check the user's profile to understand their goals and conditions
2. Personalize all advice based on their profile
3. Save any new preferences they mention

Always reference stored information to show you remember the user."""
)

print("Memory-enhanced agent created!")

Memory-enhanced agent created!


In [24]:
# Test the memory agent
TODO_STORE.clear()

result = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Hi! My user_id is user_alex. What exercise routine would you recommend for me?"
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Hi Alex! I can see from your profile that your primary goal is to improve energy levels, with better sleep as a secondary goal. You also prefer morning workouts and have mild anxiety, which I'll keep in mind for my recommendations.

Based on your goals and profile, here's a personalized exercise routine that should help boost your energy and improve your sleep quality:

## **Morning Energy-Boosting Routine (4-5 days/week)**

### **Monday, Wednesday, Friday - Energizing Cardio & Strength**
**Duration: 30-35 minutes**

1. **Warm-up (5 minutes)**
   - Light dynamic stretching
   - Arm circles and leg swings
   - Gentle marching in place

2. **Cardio Burst (15 minutes)**
   - 3 rounds of:
     - 2 minutes brisk walking or light jogging
     - 1 minute high knees or jumping jacks
     - 2 minutes recovery pace
   
3. **Strength Circuit (12 minutes)**
   - 3 rounds, 45 seconds each exercise, 15 seconds rest:
     - Bodyweight squats
     - Modified push-ups (wall or knee)
   

## Task 8: Skills - On-Demand Capabilities

**Skills** are a powerful feature for progressive capability disclosure. Instead of loading all tools upfront, agents can load specialized capabilities on demand.

### Why Skills?

1. **Context Efficiency**: Don't waste context on unused tool descriptions
2. **Specialization**: Skills can include detailed instructions for specific tasks
3. **Modularity**: Easy to add/remove capabilities
4. **Discoverability**: Agent can browse available skills

### SKILL.md Format

Skills are defined in markdown files with YAML frontmatter:

```markdown
---
name: skill-name
description: What this skill does
version: 1.0.0
tools:
  - tool1
  - tool2
---

# Skill Instructions

Detailed steps for how to use this skill...
```

In [25]:
# Let's look at the skills we created
skills_dir = Path("skills")

print("Available skills:")
for skill_dir in skills_dir.iterdir():
    if skill_dir.is_dir():
        skill_file = skill_dir / "SKILL.md"
        if skill_file.exists():
            content = skill_file.read_text()
            # Extract name and description from frontmatter
            lines = content.split("\n")
            name = ""
            desc = ""
            for line in lines:
                if line.startswith("name:"):
                    name = line.split(":", 1)[1].strip()
                if line.startswith("description:"):
                    desc = line.split(":", 1)[1].strip()
            print(f"  - {name}: {desc}")

Available skills:
  - meal-planning: Create personalized meal plans based on dietary needs and preferences
  - wellness-assessment: Assess user wellness goals and create personalized recommendations


In [26]:
# Read the wellness-assessment skill
skill_content = Path("skills/wellness-assessment/SKILL.md").read_text()
print(skill_content)

---
name: wellness-assessment
description: Assess user wellness goals and create personalized recommendations
version: 1.0.0
tools:
  - read_file
  - write_file
---

# Wellness Assessment Skill

You are conducting a comprehensive wellness assessment. Follow these steps:

## Step 1: Gather Information
Ask the user about:
- Current health goals (weight, fitness, stress, sleep)
- Any medical conditions or limitations
- Current exercise routine (or lack thereof)
- Dietary preferences and restrictions
- Sleep patterns and quality
- Stress levels and sources

## Step 2: Analyze Responses
Review the user's answers and identify:
- Primary wellness priority
- Secondary goals
- Potential barriers to success
- Existing healthy habits to build on

## Step 3: Create Assessment Report
Write a wellness assessment report to `workspace/wellness_assessment.md` containing:
- Summary of current wellness state
- Identified strengths
- Areas for improvement
- Recommended focus areas (prioritized)
- Suggeste

In [27]:
# Create a skill-aware tool
@tool
def load_skill(skill_name: str) -> str:
    """Load a skill's instructions for a specialized task.
    
    Available skills:
    - wellness-assessment: Assess user wellness and create recommendations
    - meal-planning: Create personalized meal plans
    
    Args:
        skill_name: Name of the skill to load
    
    Returns:
        Skill instructions
    """
    skill_path = Path(f"skills/{skill_name}/SKILL.md")
    if not skill_path.exists():
        available = [d.name for d in Path("skills").iterdir() if d.is_dir()]
        return f"Skill '{skill_name}' not found. Available: {', '.join(available)}"
    
    return skill_path.read_text()

print("Skill loader defined!")

Skill loader defined!


In [28]:
# Create an agent that can load and use skills
skill_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        load_skill,
        write_todos,
        update_todo,
        list_todos,
    ],
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a wellness assistant with access to specialized skills.

When a user asks for something that matches a skill:
1. Load the appropriate skill using load_skill()
2. Follow the skill's instructions carefully
3. Save outputs as specified in the skill

Available skills:
- wellness-assessment: For comprehensive wellness evaluations
- meal-planning: For creating personalized meal plans

If no skill matches, use your general wellness knowledge."""
)

print("Skill-aware agent created!")

Skill-aware agent created!


In [29]:
# Test with a skill-appropriate request
TODO_STORE.clear()

result = skill_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "I'd like a wellness assessment. I'm a 35-year-old office worker who sits most of the day, has trouble sleeping, and wants to lose 15 pounds. I'm vegetarian and have no major health conditions."
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
## Your Comprehensive Wellness Assessment

I've completed your wellness assessment and saved a detailed report. Based on your profile as a 35-year-old office worker who's vegetarian with sleep issues and a 15-pound weight loss goal, here are the key findings:

### **Your Current Wellness Score: 4.5/10**
**Target Goal: 7.5/10**

### **Top Priorities (Start These First):**

1. **Physical Activity** - Your biggest opportunity for improvement
   - Current score: 3/10 → Target: 7/10
   - Your sedentary work is impacting both weight and sleep

2. **Sleep Quality** - Critical for weight management success  
   - Current score: 4/10 → Target: 8/10
   - Poor sleep disrupts metabolism and hunger hormones

### **Your Strengths to Build On:**
✅ Vegetarian diet (great nutritional foundation)
✅ No major health conditions  
✅ Clear, specific goals
✅ Proactive wellness mindset

### **Start Today - 3 Immediate Actions:**

1. **Set hourly movement alarms** - 2-minute breaks to combat sit

## Task 9: Using deepagents-cli

The `deepagents-cli` provides an interactive terminal interface for working with Deep Agents.

### Installation

```bash
uv pip install deepagents-cli
# or
pip install deepagents-cli
```

### Key Features

| Feature | Description |
|---------|-------------|
| **Interactive Sessions** | Chat with your agent in the terminal |
| **Conversation Resume** | Pick up where you left off |
| **Human-in-the-Loop** | Approve or reject agent actions |
| **File System Access** | Agent can read/write to your filesystem |
| **Remote Sandboxing** | Run in isolated Docker containers |

### Basic Usage

```bash
# Start an interactive session
deepagents

# Resume a previous conversation
deepagents --resume

# Use a specific model
deepagents --model openai:gpt-4o

# Enable human-in-the-loop approval
deepagents --approval-mode full
```

### Example Session

```
$ deepagents

Welcome to Deep Agents CLI!

You: Create a 7-day meal plan for a vegetarian athlete

Agent: I'll create a comprehensive meal plan for you. Let me:
1. Research vegetarian athlete nutrition needs
2. Design balanced daily menus
3. Save the plan to a file

[Agent uses tools...]

Agent: I've created your meal plan! You can find it at:
workspace/vegetarian_athlete_meal_plan.md

You: /exit
```

In [30]:
# Check if CLI is installed
import subprocess

try:
    result = subprocess.run(["deepagents", "--version"], capture_output=True, text=True)
    print(f"deepagents-cli version: {result.stdout.strip()}")
except FileNotFoundError:
    print("deepagents-cli not installed. Install with:")
    print("  uv pip install deepagents-cli")
    print("  # or")
    print("  pip install deepagents-cli")

deepagents-cli version: deepagents 0.0.17


### Try It Yourself!

After installing the CLI, try these commands in your terminal:

```bash
# Basic interactive session
deepagents

# With a specific working directory
deepagents --workdir ./workspace

# See all options
deepagents --help
```

Sample prompts to try:
1. "Create a weekly workout plan and save it to a file"
2. "Research the health benefits of meditation and summarize in a report"
3. "Analyze my current diet and suggest improvements" (then provide details)

## Task 10: Building a Complete Deep Agent System

Now let's bring together all four elements to build a comprehensive "Wellness Coach" system:

1. **Planning**: Track multi-week wellness programs
2. **Context Management**: Store session notes and progress
3. **Subagent Spawning**: Delegate to specialists (exercise, nutrition, mindfulness)
4. **Long-term Memory**: Remember user preferences and history

In [31]:
# Define specialized wellness subagents
# Subagents inherit the backend from the parent, so they use the same workspace
exercise_specialist = {
    "name": "exercise-specialist",
    "description": "Expert in exercise science, workout programming, and physical fitness. Use for exercise-related questions and plan creation.",
    "system_prompt": """You are an exercise specialist with expertise in:
- Workout programming for different fitness levels
- Exercise form and safety
- Progressive overload principles
- Recovery and injury prevention

Always consider the user's fitness level and any physical limitations.
Provide clear, actionable exercise instructions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

nutrition_specialist = {
    "name": "nutrition-specialist",
    "description": "Expert in nutrition science, meal planning, and dietary optimization. Use for food-related questions and meal plans.",
    "system_prompt": """You are a nutrition specialist with expertise in:
- Macro and micronutrient balance
- Meal planning and preparation
- Dietary restrictions and alternatives
- Nutrition timing for performance

Always respect dietary restrictions and preferences.
Focus on practical, achievable meal suggestions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

mindfulness_specialist = {
    "name": "mindfulness-specialist",
    "description": "Expert in stress management, sleep optimization, and mental wellness. Use for stress, sleep, and mental health questions.",
    "system_prompt": """You are a mindfulness and mental wellness specialist with expertise in:
- Stress reduction techniques
- Sleep hygiene and optimization
- Meditation and breathing exercises
- Work-life balance strategies

Be supportive and non-judgmental.
Provide practical techniques that can be implemented immediately.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

print("Specialist subagents defined!")

Specialist subagents defined!


In [32]:
# Create the Wellness Coach coordinator
wellness_coach = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        # Planning
        write_todos,
        update_todo,
        list_todos,
        # Long-term Memory
        get_user_profile,
        save_user_preference,
        # Skills
        load_skill,
    ],
    backend=filesystem_backend,  # All file ops go to workspace
    subagents=[exercise_specialist, nutrition_specialist, mindfulness_specialist],
    system_prompt="""You are a Personal Wellness Coach that coordinates comprehensive wellness programs.

## Your Role
- Understand each user's unique goals, constraints, and preferences
- Create personalized, multi-week wellness programs
- Coordinate between exercise, nutrition, and mindfulness specialists
- Track progress and adapt recommendations

## Workflow
1. **Initial Assessment**: Get user profile and understand their situation
2. **Planning**: Create a todo list for the program components
3. **Delegation**: Use specialists for domain-specific content:
   - exercise-specialist: Workout plans and fitness guidance
   - nutrition-specialist: Meal plans and dietary advice
   - mindfulness-specialist: Stress and sleep optimization
4. **Integration**: Combine specialist outputs into a cohesive program
5. **Documentation**: Save all plans and recommendations to files

## Important
- Always check user profile first for context
- Respect any medical conditions or dietary restrictions
- Provide clear, actionable recommendations
- Save progress to files so users can reference later"""
)

print("Wellness Coach created with all 4 Deep Agent elements!")

Wellness Coach created with all 4 Deep Agent elements!


In [33]:
# Test the complete system
TODO_STORE.clear()

result = wellness_coach.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex. I'd like you to create a 2-week wellness program for me.

I want to focus on:
1. Building a consistent exercise routine (I can exercise 3x per week for 30 mins)
2. Improving my diet (remember I'm vegetarian)
3. Better managing my work stress and improving my sleep

Please create comprehensive plans for each area and save them as separate files I can reference."""
    }]
})

print("Wellness Coach response:")
print(result["messages"][-1].content)

Wellness Coach response:
## Your Complete 2-Week Wellness Program is Ready! 🎉

I've successfully created a comprehensive, personalized wellness program that addresses all your goals. Here's what you now have access to:

### **📁 Your Reference Files:**
1. **`/alex_complete_wellness_program.md`** - Your master integrated program
2. **`/alex_exercise_plan.md`** - Detailed 3x/week exercise routine
3. **`/alex_vegetarian_meal_plan.md`** - Complete 2-week meal plan with recipes
4. **`/alex_stress_sleep_plan.md`** - Stress management and sleep optimization

### **🎯 Program Highlights:**
- **Exercise:** 3x/week morning workouts (Cardio/Core, Strength, Yoga/Pilates)
- **Nutrition:** Energy-boosting vegetarian meals with meal prep guidance
- **Stress Management:** Workplace techniques and anxiety tools for mild anxiety
- **Sleep:** Complete sleep hygiene protocol with evening routines

### **🔄 How Everything Connects:**
Your program is designed as an integrated system where:
- Morning exercise b

In [34]:
# Review what was created
print("=" * 60)
print("FINAL TODO STATUS")
print("=" * 60)
print(list_todos.invoke({}))

print("\n" + "=" * 60)
print("GENERATED FILES")
print("=" * 60)
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

FINAL TODO STATUS
✅ [todo_1] Create comprehensive exercise plan (completed)
✅ [todo_3] Develop vegetarian meal plan (completed)
✅ [todo_5] Design stress management and sleep optimization plan (completed)
✅ [todo_7] Integrate all plans into a cohesive program (completed)
✅ [todo_9] Save all plans as separate reference files (completed)

GENERATED FILES
  [FILE] alex_2week_wellness_program_overview.md (3502 bytes)
  [FILE] alex_complete_wellness_program.md (7873 bytes)
  [FILE] alex_exercise_plan.md (4793 bytes)
  [FILE] alex_stress_sleep_plan.md (6389 bytes)
  [FILE] alex_vegetarian_energy_boosting_foods.txt (603 bytes)
  [FILE] alex_vegetarian_meal_plan.md (5924 bytes)
  [FILE] alex_vegetarian_meal_plan_reference.txt (1097 bytes)
  [FILE] alex_vegetarian_meal_plan_week_1.txt (2629 bytes)
  [FILE] alex_vegetarian_meal_plan_week_2.txt (2384 bytes)
  [FILE] alex_vegetarian_nutritional_guidelines.txt (1245 bytes)
  [FILE] alex_vegetarian_shopping_list_week_1.txt (940 bytes)
  [FILE] alex_v

In [35]:
# Read one of the generated files
files = list(WORKSPACE.glob("*.md"))
if files:
    print(f"\nContents of {files[0].name}:")
    print("=" * 60)
    print(files[0].read_text()[:2000] + "..." if len(files[0].read_text()) > 2000 else files[0].read_text())


Contents of alex_complete_wellness_program.md:
# Alex's Complete 2-Week Wellness Program

## Program Overview
This comprehensive wellness program integrates exercise, nutrition, and stress/sleep management to help you:
- Build a consistent exercise routine (3x/week, 30 minutes)
- Optimize your vegetarian diet for energy
- Manage work stress and improve sleep quality
- Increase overall energy levels

## Your Personal Profile
- **Name:** Alex
- **Goals:** Improve energy levels, better sleep, consistent exercise routine
- **Dietary Requirements:** Vegetarian
- **Health Considerations:** Mild anxiety
- **Preferences:** Morning exercise, detailed guidance
- **Schedule:** 3x/week exercise, 30 minutes per session

## Daily Integrated Schedule

### Monday - Exercise Day
**Morning (6:30-7:30 AM):**
- Wake-up breathing & intention setting (10 min)
- Pre-workout snack: Banana with almond butter
- **EXERCISE:** Cardio & Core workout (30 min)
- Post-workout: Protein smoothie

**Meals:**
- Breakfas

---
## ❓ Question #3:

What are the key considerations when designing **subagent configurations**?

Consider:
- When should subagents share tools vs have distinct tools?
- How do you decide which model to use for each subagent?
- What's the right granularity for subagent specialization?

##### Answer:
*Your answer here*
Share tools when there is low risk involved. That way there will be consistent output, less rules to manage. Distinct tools when cost is involved, to avoid agent calling wrong tool, tool may leak sensitive information. Use smaller models for fast responses, low cost, easy output to verify, low errors vs use a stronger model when reasoning matters, ambiguity involved and mistakes are unsafe and expensive. The function of a particular query is what determines the subagents with clean input/ output responses. When there is no confusion or overlapping within tools it gets easier to subagents, because every handoffs costs tokens, latency. 


## ❓ Question #4:

For a **production wellness application** using Deep Agents, what would you need to add?

Consider:
- Safety guardrails for health advice
- Persistent storage (not in-memory)
- Multi-user support and isolation
- Monitoring and observability
- Cost management with subagents

##### Answer:
*Your answer here*
Safety boundaries, storage, cost controls. Deepagents do depend on LLMs, so the model has to be powerful. Safety guardrails : Tools that are safe for the agents must be exposed. There must be a deterministic layer to blocks when diagnosis output is present. How about a bookkeeper task : have a safety subagent just to keep track of the responses. Deepagents includes a filesystem backend for context management/offloading. Add - event log(what agent said, what it retrieved, what tools were called). Trace everything tokens used, tool calls, retrieval, latency, failures because deepagents can run long, beacuse of its nature. Cost management can be challenging and can be overwhelming at the same time, so we can add a limit on tokens, tool calls.


---
## 🏗️ Activity #2: Build a Wellness Coach Agent

Build your own wellness coach that uses all 4 Deep Agent elements.

### Requirements:
1. **Planning**: Create todos for a 30-day wellness challenge
2. **Context Management**: Store daily check-in notes
3. **Subagents**: At least 2 specialized subagents
4. **Memory**: Remember user preferences across interactions

### Challenge:
Create a "30-Day Wellness Challenge" system that:
- Generates a personalized 30-day plan
- Tracks daily progress
- Adapts recommendations based on feedback
- Saves a weekly summary report

In [36]:
### YOUR CODE HERE ###

from pathlib import Path
from datetime import datetime, timedelta
import json
import random

# -----------------------------
# Step 1: Define your subagent configurations
# -----------------------------

class MovementCoach:
    """Subagent #1: creates movement/exercise suggestions."""
    def recommend(self, day, prefs):
        level = prefs.get("fitness_level", "beginner")
        if level == "beginner":
            options = [
                "10–15 min walk",
                "10 min gentle yoga/stretching",
                "5 min mobility + 5 min walk",
            ]
        else:
            options = [
                "20–30 min brisk walk",
                "15 min strength (bodyweight)",
                "20 min yoga + core",
            ]
        return random.choice(options)


class MindfulnessCoach:
    """Subagent #2: creates stress/mindfulness suggestions."""
    def recommend(self, day, prefs):
        options = [
            "5 min box breathing (4-4-4-4)",
            "5 min mindfulness meditation",
            "Write 3 lines of journaling (stress + reframe)",
            "2 min gratitude list + 3 min breathing",
        ]
        return random.choice(options)


# -----------------------------
# Step 2: Create any additional tools you need
# -----------------------------

def load_memory(memory_path="wellness_memory.json"):
    p = Path(memory_path)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    return {
        "user_preferences": {},
        "daily_checkins": [],  # list of dicts
        "weekly_summaries": [] # list of dicts
    }

def save_memory(memory, memory_path="wellness_memory.json"):
    Path(memory_path).write_text(json.dumps(memory, indent=2), encoding="utf-8")

def today_str():
    return datetime.now().strftime("%Y-%m-%d")

def render_todos_md(todos):
    lines = ["## Planning Todos\n"]
    for t in todos:
        checkbox = "[x]" if t["status"] == "DONE" else "[ ]"
        lines.append(f"- {checkbox} **{t['status']}** — {t['task']}")
    return "\n".join(lines)

def save_weekly_summary_md(summary_text, week_num, out_dir="weekly_reports"):
    Path(out_dir).mkdir(exist_ok=True)
    out_path = Path(out_dir) / f"week_{week_num}_summary.md"
    out_path.write_text(summary_text, encoding="utf-8")
    return str(out_path)


# -----------------------------
# Step 3: Build the main coordinator agent
# -----------------------------

class WellnessCoordinator:
    """
    Main deep agent that uses:
    - Planning (todos)
    - Context management (daily check-ins)
    - Subagents (movement + mindfulness)
    - Memory (preferences saved to JSON)
    """

    def __init__(self, memory_path="wellness_memory.json"):
        self.memory_path = memory_path
        self.memory = load_memory(memory_path)

        self.movement = MovementCoach()
        self.mindfulness = MindfulnessCoach()

    def set_preferences(self, prefs: dict):
        self.memory["user_preferences"].update(prefs)
        save_memory(self.memory, self.memory_path)

    def create_30_day_plan(self):
        prefs = self.memory["user_preferences"]

        # Planning todos
        todos = [
            {"task": "Collect user preferences", "status": "DONE" if prefs else "TODO"},
            {"task": "Generate 30-day wellness plan", "status": "DOING"},
            {"task": "Store plan in memory", "status": "TODO"},
            {"task": "Initialize daily check-in tracker", "status": "TODO"},
        ]

        start = datetime.now().date()
        plan = []

        for i in range(30):
            day_date = start + timedelta(days=i)
            day_num = i + 1

            movement_task = self.movement.recommend(day_num, prefs)
            mindfulness_task = self.mindfulness.recommend(day_num, prefs)

            plan.append({
                "day": day_num,
                "date": str(day_date),
                "movement": movement_task,
                "mindfulness": mindfulness_task,
                "nutrition": "Drink 6–8 glasses of water + include 1 fruit/veg serving",
                "sleep": "Aim for consistent bedtime; reduce screens 30 min before bed",
            })

        todos[1]["status"] = "DONE"
        todos[2]["status"] = "DOING"

        self.memory["plan_30_days"] = plan
        save_memory(self.memory, self.memory_path)

        todos[2]["status"] = "DONE"
        todos[3]["status"] = "DONE"

        return todos, plan

    def daily_checkin(self, day_num: int, mood: str, stress_level: int, completed: dict, notes: str):
        """
        completed example:
        {"movement": True, "mindfulness": False, "nutrition": True, "sleep": False}
        """
        prefs = self.memory["user_preferences"]
        plan = self.memory.get("plan_30_days", [])

        if not plan or day_num < 1 or day_num > 30:
            raise ValueError("Plan missing or invalid day number.")

        day_plan = plan[day_num - 1]

        checkin = {
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "day": day_num,
            "date": day_plan["date"],
            "mood": mood,
            "stress_level": stress_level,
            "completed": completed,
            "notes": notes
        }

        # Context Management: store daily notes
        self.memory["daily_checkins"].append(checkin)

        # Adaptation logic (simple + clear)
        adaptation = []

        if stress_level >= 7:
            adaptation.append("Stress is high → recommend shorter movement + extra breathing.")
            day_plan["movement"] = "10 min gentle walk (low intensity)"
            day_plan["mindfulness"] = "7 min breathing + 3 min journaling (reframe)"

        if not completed.get("mindfulness", True):
            adaptation.append("Mindfulness not completed → simplify tomorrow’s mindfulness task.")
            # Lighten tomorrow if exists
            if day_num < 30:
                plan[day_num]["mindfulness"] = "3 min box breathing (easy mode)"

        if not completed.get("movement", True):
            adaptation.append("Movement not completed → reduce tomorrow’s movement goal.")
            if day_num < 30:
                plan[day_num]["movement"] = "10 min walk (easy mode)"

        # Save updated plan + memory
        self.memory["plan_30_days"] = plan
        save_memory(self.memory, self.memory_path)

        return checkin, adaptation

    def weekly_summary(self, week_num: int):
        """
        Creates and saves a weekly summary report.
        week_num: 1..4 (week 4 includes days 22-28; remaining days handled separately)
        """
        if week_num < 1 or week_num > 5:
            raise ValueError("week_num should be 1..5")

        start_day = (week_num - 1) * 7 + 1
        end_day = min(week_num * 7, 30)

        checkins = [c for c in self.memory["daily_checkins"] if start_day <= c["day"] <= end_day]

        if not checkins:
            summary = f"# Week {week_num} Summary\n\nNo check-ins recorded for days {start_day}-{end_day}.\n"
        else:
            avg_stress = sum(c["stress_level"] for c in checkins) / len(checkins)

            completed_counts = {"movement": 0, "mindfulness": 0, "nutrition": 0, "sleep": 0}
            for c in checkins:
                for k in completed_counts:
                    if c["completed"].get(k):
                        completed_counts[k] += 1

            summary = []
            summary.append(f"# Week {week_num} Wellness Summary\n")
            summary.append(f"**Days covered:** {start_day}–{end_day}")
            summary.append(f"**Check-ins recorded:** {len(checkins)}")
            summary.append(f"**Average stress level:** {avg_stress:.1f}\n")

            summary.append("## Completion Stats\n")
            for k, v in completed_counts.items():
                summary.append(f"- {k}: {v}/{len(checkins)}")

            summary.append("\n## Key Notes\n")
            for c in checkins[-3:]:
                summary.append(f"- Day {c['day']}: mood={c['mood']}, stress={c['stress_level']} — {c['notes']}")

            summary.append("\n## Next Week Recommendation\n")
            if avg_stress >= 7:
                summary.append("- Keep goals lighter and emphasize breathing + sleep routine.")
            else:
                summary.append("- Gradually increase consistency and keep habits simple.")

            summary = "\n".join(summary)

        # Save summary
        out_path = save_weekly_summary_md(summary, week_num)
        self.memory["weekly_summaries"].append({
            "week": week_num,
            "generated": datetime.now().isoformat(timespec="seconds"),
            "path": out_path
        })
        save_memory(self.memory, self.memory_path)

        return summary, out_path

# -----------------------------
# Step 4: Test with a user creating their 30-day challenge
# -----------------------------

coach = WellnessCoordinator()

# Memory: store preferences
coach.set_preferences({
    "fitness_level": "beginner",
    "preferred_time": "morning",
    "goal": "reduce stress + build daily consistency",
})

todos, plan = coach.create_30_day_plan()

print("TODOS:")
print(render_todos_md(todos))
print("\nDay 1 Plan:")
print(plan[0])

# -----------------------------
# Step 5: Simulate a daily check-in and adaptation
# -----------------------------

checkin, adaptation = coach.daily_checkin(
    day_num=1,
    mood="anxious",
    stress_level=8,
    completed={"movement": True, "mindfulness": False, "nutrition": True, "sleep": False},
    notes="Busy day, couldn't focus long. Breathing helped a bit."
)

print("\nCheck-in recorded:")
print(checkin)

print("\nAdaptations:")
for a in adaptation:
    print("-", a)

# Generate a weekly summary (week 1)
summary, path = coach.weekly_summary(week_num=1)
print("\nWeekly summary saved to:", path)
print("\nSummary preview:\n", summary[:400])



TODOS:
## Planning Todos

- [x] **DONE** — Collect user preferences
- [x] **DONE** — Generate 30-day wellness plan
- [x] **DONE** — Store plan in memory
- [x] **DONE** — Initialize daily check-in tracker

Day 1 Plan:
{'day': 1, 'date': '2026-02-22', 'movement': '10–15 min walk', 'mindfulness': '5 min box breathing (4-4-4-4)', 'nutrition': 'Drink 6–8 glasses of water + include 1 fruit/veg serving', 'sleep': 'Aim for consistent bedtime; reduce screens 30 min before bed'}

Check-in recorded:
{'timestamp': '2026-02-22T15:08:13', 'day': 1, 'date': '2026-02-22', 'mood': 'anxious', 'stress_level': 8, 'completed': {'movement': True, 'mindfulness': False, 'nutrition': True, 'sleep': False}, 'notes': "Busy day, couldn't focus long. Breathing helped a bit."}

Adaptations:
- Stress is high → recommend shorter movement + extra breathing.
- Mindfulness not completed → simplify tomorrow’s mindfulness task.

Weekly summary saved to: weekly_reports/week_1_summary.md

Summary preview:
 # Week 1 Wellness

---
## Summary

In this session, we explored **Deep Agents** and their four key elements:

| Element | Purpose | Implementation |
|---------|---------|----------------|
| **Planning** | Track complex tasks | `write_todos`, `update_todo`, `list_todos` |
| **Context Management** | Handle large contexts | File system tools, automatic offloading |
| **Subagent Spawning** | Delegate to specialists | `task` tool with custom configs |
| **Long-term Memory** | Remember across sessions | LangGraph Store integration |

### Key Takeaways:

1. **Deep Agents handle complexity** - Unlike simple tool loops, they can manage long-horizon, multi-step tasks
2. **Planning is context engineering** - Todo lists and files aren't just organization—they're extended memory
3. **Subagents prevent context bloat** - Delegation keeps the main agent focused and efficient
4. **Skills enable progressive disclosure** - Load capabilities on-demand instead of upfront
5. **The CLI makes interaction natural** - Interactive sessions with conversation resume

### Deep Agents vs Traditional Agents

| Aspect | Traditional Agent | Deep Agent |
|--------|-------------------|------------|
| Task complexity | Simple, single-step | Complex, multi-step |
| Context management | All in conversation | Files + summaries |
| Delegation | None | Subagent spawning |
| Memory | Within thread | Across sessions |
| Planning | Implicit | Explicit (todos) |

### When to Use Deep Agents

**Use Deep Agents when:**
- Tasks require multiple steps or phases
- Context would overflow in a simple loop
- Specialization would improve quality
- Users need to resume sessions
- Long-term memory is valuable

**Use Simple Agents when:**
- Tasks are straightforward Q&A
- Single tool call suffices
- Context fits easily
- No need for persistence

### Further Reading

- [Deep Agents Documentation](https://docs.langchain.com/oss/python/deepagents/overview)
- [Deep Agents GitHub](https://github.com/langchain-ai/deepagents)
- [Context Management Blog Post](https://www.blog.langchain.com/context-management-for-deepagents/)
- [Building Multi-Agent Applications](https://www.blog.langchain.com/building-multi-agent-applications-with-deep-agents/)
- [LangGraph Memory Concepts](https://langchain-ai.github.io/langgraph/concepts/memory/)